# Fixed-Size Text Chunking 

# 1. What is Chunking?

AI models can only read a limited amount of text at a time. A long document, like a 30-page PDF, is too big to hand over in one piece. So we **split the document into smaller pieces called chunks**. Later, when someone asks a question, the system finds only the most relevant chunks and gives them to the AI.

> **Analogy:** Think of a textbook with an index. You don't read the whole book to answer one question. You jump to the right pages. Chunks are those "pages" for an AI.

---

# 2. What is Fixed-Size Chunking?

Fixed-size chunking cuts a document into pieces of **equal length**, no matter what the text says. It doesn't care about sentences, paragraphs or topics. It just counts and cuts.

> **Analogy:** Slicing a loaf of bread into equal slices. Every slice is the same thickness, even if you cut through the middle of a raisin.

### Example
Original text:
`Machine learning is a field of study in artificial intelligence concerned with algorithms that learn from data.`

With **chunk size = 40 characters** and **overlap = 0**:

| Chunk | Text |
|-------|------|
| 1 | `Machine learning is a field of study in ` |
| 2 | `artificial intelligence concerned with a` |
| 3 | `lgorithms that learn from data.` |

Notice chunk 3 begins with `lgorithms`: the word "algorithms" was cut in half. This is the main weakness of fixed-size chunking, and it is why **overlap** exists.

---

# 3. Why Do We Need Chunking?

| Reason | Explanation |
|--------|-------------|
| **Context window limit** | AI models can only process a limited amount of text per request. |
| **Better search accuracy** | Searching small chunks finds the exact relevant part instead of a whole document. |
| **Lower cost and faster answers** | Sending 3 small chunks to an AI is cheaper and faster than sending the whole document. |
| **Less noise** | Irrelevant text can confuse the AI. Small, focused chunks reduce that. |
| **Foundation of RAG** | Retrieval-Augmented Generation (chatbots that answer from your documents) depends on chunking. |

---

# 4. Key Terms: Chunk Size and Overlap

## Chunk Size
The length of each chunk, measured in **characters, words or tokens**.

- **Small chunks (e.g. 200 characters):** very precise, but may lose context.
- **Large chunks (e.g. 2000 characters):** keep more context, but contain more irrelevant text and can dilute the answer.

## Overlap
The amount of text **repeated** between two neighbouring chunks.

Example: chunk size = 100, overlap = 20. Chunk 1 covers characters 0 to 100. Chunk 2 covers characters **80** to 180, so the last 20 characters of chunk 1 appear again at the start of chunk 2.

```
Chunk 1: [==========================]
Chunk 2:                      [==========================]
                              ^^^^^^ overlap (shared text)
```

**Why overlap?** If an important sentence is cut at a boundary, the overlap keeps a copy of it in the next chunk, so its meaning isn't lost.

**Trade-off:** more overlap means better context, but also more chunks, more storage, more duplicate content and higher cost.

**Rule of thumb:** overlap is usually **10% to 20%** of the chunk size.

---


---

# 5. Pros and Cons

## Pros
- **Very simple** to understand and implement (a few lines of code)
- **Fast** to run, even on huge document collections
- **Predictable:** every chunk has a known size, which makes memory and cost easy to estimate
- **Works on any text** in any language or format, with no special rules needed
- **Good baseline:** a starting point to compare smarter methods against

## Cons
- **Ignores meaning:** it can cut in the middle of a sentence, a table or an idea
- **Mixed topics:** one chunk may contain the end of one topic and the start of another
- **Needs tuning:** the best size and overlap are different for every document
- **Overlap adds duplication**, which increases storage and cost
- **Weak on structured content** like tables, code or documents with headings

---

# 6. Which Documents Is It Useful For?

## Works well for
- **Plain, continuous text:** articles, blogs, essays, news, Wikipedia pages (like our Machine Learning PDF)
- **Very large collections** where speed matters more than perfection
- **Uniform documents:** transcripts, chat logs, emails, reports with consistent style
- **Quick prototypes** and baseline experiments

## Not ideal for
- **Structured documents** with clear headings and sections (legal contracts, research papers): better to split by section
- **Tables, spreadsheets, code:** cutting through a row or a function destroys meaning
- **Short documents** like FAQs, where each Q&A is already a natural chunk
- **Documents where every sentence is critical** (medical or legal text), because a bad cut can change the meaning

---

# 7. Choosing Chunk Size and Overlap

There is **no universal best value**. It depends on the document and the type of questions asked.

| Situation | Suggested direction |
|-----------|---------------------|
| Questions about specific facts ("Who coined the term ML?") | Smaller chunks (200 to 500 characters) |
| Questions needing explanation or context ("How does supervised learning work?") | Larger chunks (800 to 1500 characters) |
| Dense text where ideas span many sentences | More overlap (15% to 20%) |
| Text where each paragraph is self-contained | Less overlap (0% to 10%) |

**Our approach in this notebook:** instead of guessing, we will pick a **minimum and maximum** for both chunk size and overlap, try random values between them, and measure which combination retrieves the most relevant chunks for our test questions.

---

# 8. Summary

- Chunking splits long documents into small pieces so AI can search and read them efficiently.
- Fixed-size chunking cuts by length only: simple, fast and predictable, but blind to meaning.
- **Chunk size** controls precision versus context. **Overlap** protects against bad cut points.
- The best values depend on the document, so we find them by **experimenting**.

## Part 1: Setup and Imports

First, we'll import all the libraries we need.

In [1]:
# Install required libraries
import subprocess
import sys

# Install PyPDF2 if not already installed
try:
    import PyPDF2
    print("✅ PyPDF2 already installed")
except ImportError:
    print("📦 Installing PyPDF2...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "PyPDF2"])
    print("✅ PyPDF2 installed successfully")

✅ PyPDF2 already installed


In [2]:
# Import all libraries
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple
from collections import defaultdict
import warnings

# For PDF reading
import PyPDF2
from pathlib import Path

# Styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"\n📊 NumPy version: {np.__version__}")
print(f"📊 Pandas version: {pd.__version__}")
print(f"📊 Matplotlib version: {plt.matplotlib.__version__}")

✅ All libraries imported successfully!

📊 NumPy version: 2.2.6
📊 Pandas version: 2.3.3
📊 Matplotlib version: 3.10.9


## Step 2: Loading the PDF and Extracting Text

Before we can chunk anything, we need the raw text from the PDF.

**What we do here:**
1. Read every page of the PDF and pull out its text
2. Clean the text (remove citation numbers like `[12]` and the long References list)
3. Check basic statistics: pages, characters, words

**Why clean?** The References section is just a list of links and book titles. If we keep it, our chunks would be
full of useless text and our experiment results would be unfair.

In [3]:

pdf_path = 'Data/Machine_learning.pdf'

# Check if file exists
if os.path.exists(pdf_path):
    print(f"✅ PDF file found: {pdf_path}")
else:
    print(f"❌ PDF file not found at: {pdf_path}")
    print(f"\n📍 Please update the pdf_path variable with your actual file location.")
    print(f"   Example: '/path/to/your/Machine_learning.pdf'")

✅ PDF file found: Data/Machine_learning.pdf


In [4]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract text from PDF file
    
    Args:
        pdf_path: Path to PDF file
    
    Returns:
        Extracted text as string
    """
    text = ""
    
    try:
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            num_pages = len(pdf_reader.pages)
            
            print(f"📄 PDF Information:")
            print(f"   Total pages: {num_pages}")
            
            # Extract text from each page
            for page_num, page in enumerate(pdf_reader.pages, 1):
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
                    
                # Show progress
                if page_num % 10 == 0:
                    print(f"   ✓ Processed {page_num}/{num_pages} pages")
            
            print(f"   ✓ Processed {num_pages}/{num_pages} pages")
            
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        return ""
    
    return text

# Extract text
print("🔄 Extracting text from PDF...\n")
document = extract_text_from_pdf(pdf_path)

print(f"\n✅ Text extraction complete!")
print(f"   Total characters: {len(document)}")
print(f"   Total words: {len(document.split())}")

🔄 Extracting text from PDF...

📄 PDF Information:
   Total pages: 44
   ✓ Processed 10/44 pages
   ✓ Processed 20/44 pages
   ✓ Processed 30/44 pages
   ✓ Processed 40/44 pages
   ✓ Processed 44/44 pages

✅ Text extraction complete!
   Total characters: 149500
   Total words: 18511


## Step 3: Cleaning Text

In [5]:
# ---------- Step 2b: Clean the extracted text (and save to .txt) ----------


# ===== OUTPUT FILES =====
CLEAN_TXT_PATH = "cleaned_document.txt"
RAW_TXT_PATH   = "raw_document.txt"     # None kar do agar raw save nahi karna
# ========================

raw_document = document   # save the original so we can compare later

def clean_text(text: str) -> str:
    """Keep only the article body and remove PDF noise."""

    # 1. Cut everything after the last sentence of the article body
    end_marker = re.search(r"and\s+parameter\s+sharing\.", text)
    if end_marker:
        text = text[:end_marker.end()]
    else:
        print("⚠️ End marker not found: references were NOT removed!")

    # 2. Remove citation markers like [12], [27]:488, [27]:708–710,755
    text = re.sub(r"\[\d+\](?::\d+(?:[–-]\d+)?(?:,\d+)*)?", "", text)

    # 3. Turn all newlines/tabs/multiple spaces into a single space
    text = re.sub(r"\s+", " ", text)

    # 4. Remove the space before punctuation: "data ." -> "data."
    text = re.sub(r"\s+([.,;:])", r"\1", text)

    return text.strip()

document = clean_text(raw_document)

# ---------- Save to .txt files ----------
Path(CLEAN_TXT_PATH).write_text(document, encoding="utf-8")
print(f"💾 Cleaned text saved -> {Path(CLEAN_TXT_PATH).resolve()}")

if RAW_TXT_PATH:
    Path(RAW_TXT_PATH).write_text(raw_document, encoding="utf-8")
    print(f"💾 Raw text saved     -> {Path(RAW_TXT_PATH).resolve()}")

# ---------- Before vs After ----------
comparison = pd.DataFrame({
    "Version": ["Raw (before)", "Cleaned (after)"],
    "Characters": [len(raw_document), len(document)],
    "Words": [len(raw_document.split()), len(document.split())],
})
print()
print(comparison.to_string(index=False))
print(f"\nRemoved: {round((1 - len(document) / len(raw_document)) * 100, 1)}%")

# ---------- Sanity checks ----------
print("\n🔎 Checks:")
print("  'doi:' left in text     ->", "doi:" in document)
print("  'ISBN' left in text     ->", "ISBN" in document)
print("  '[' brackets remaining  ->", document.count("["))

# Saved file ko wapas parh kar verify karo
saved = Path(CLEAN_TXT_PATH).read_text(encoding="utf-8")
print("  saved file == document  ->", saved == document)

print("\n📄 First 300 characters:\n", document[:300])
print("\n📄 Last 300 characters:\n", document[-300:])

💾 Cleaned text saved -> /home/as/Documents/week8/Fixed_chunking/cleaned_document.txt
💾 Raw text saved     -> /home/as/Documents/week8/Fixed_chunking/raw_document.txt

        Version  Characters  Words
   Raw (before)      149500  18511
Cleaned (after)       65050   9492

Removed: 56.5%

🔎 Checks:
  'doi:' left in text     -> False
  'ISBN' left in text     -> False
  '[' brackets remaining  -> 0
  saved file == document  -> True

📄 First 300 characters:
 Machine learning Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Statistics and mathematical opt

📄 Last 300 characters:
 achine learning can be achieved through variou s techniques, such as hardware acceleration, approximate computing, and model optimization. Common optimization techniques include pruning, quantisation, knowledge dis

# Step 4: The Fixed-Size Chunking Function

Now that the text is clean, we can cut it into chunks.

**How it works:**
1. Start at position 0 of the document
2. Take `chunk_size` characters, and that is chunk 1
3. Move forward by `chunk_size - overlap` characters (not the full chunk size)
4. Take the next `chunk_size` characters, and that is chunk 2
5. Repeat until the document ends

**Example:** chunk_size = 100, overlap = 20 -> step = 80

| Chunk | Covers characters |
|-------|-------------------|
| 1 | 0 to 100 |
| 2 | 80 to 180 |
| 3 | 160 to 260 |

The 20 characters shared between neighbours are the **overlap**.

In [6]:
# ---------- Step 3: Fixed-size chunking ----------
def fixed_size_chunking(text: str, chunk_size: int, overlap: int) -> List[str]:
    """Split text into fixed-size character chunks with overlap."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    step = chunk_size - overlap
    chunks = []
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        if start + chunk_size >= len(text):   # last chunk reached
            break
    return chunks


# ---------- Live demo on a tiny text (great for the presentation) ----------
demo_text = "Machine learning is a field of study in artificial intelligence concerned with algorithms that learn from data."

print("=== No overlap (chunk_size=40, overlap=0) ===")
for i, c in enumerate(fixed_size_chunking(demo_text, 40, 0), 1):
    print(f"Chunk {i}: |{c}|")

print("\n=== With overlap (chunk_size=40, overlap=10) ===")
for i, c in enumerate(fixed_size_chunking(demo_text, 40, 10), 1):
    print(f"Chunk {i}: |{c}|")

# ---------- Now on the real cleaned document ----------
chunks = fixed_size_chunking(document, chunk_size=500, overlap=50)
lengths = [len(c) for c in chunks]

print(f"\nReal document -> {len(chunks)} chunks (size=500, overlap=50)")
print(f"Average length: {np.mean(lengths):.0f} | Min: {min(lengths)} | Max: {max(lengths)}")
print(f"\nChunk 1:\n{chunks[0]}")
print(f"\nChunk 2 (notice it starts with the last 50 chars of chunk 1):\n{chunks[1]}")

=== No overlap (chunk_size=40, overlap=0) ===
Chunk 1: |Machine learning is a field of study in |
Chunk 2: |artificial intelligence concerned with a|
Chunk 3: |lgorithms that learn from data.|

=== With overlap (chunk_size=40, overlap=10) ===
Chunk 1: |Machine learning is a field of study in |
Chunk 2: | study in artificial intelligence concer|
Chunk 3: |nce concerned with algorithms that learn|
Chunk 4: |that learn from data.|

Real document -> 145 chunks (size=500, overlap=50)
Average length: 498 | Min: 250 | Max: 500

Chunk 1:
Machine learning Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Statistics and mathematical optimisation methods compose the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) through unsuperv

# Step 5: Test Questions (How We Will Measure "Best")

To decide which chunk size and overlap work best, we need a fair test.

**Idea:**
1. We write 20 questions whose answers we know are inside the document
2. For each question we also store a short **answer phrase** (e.g. "arthur samuel")
3. Later, for every chunk setting, we check: does the retrieved chunk contain the answer phrase?

**Why this works:** if the chunk contains the answer phrase, the AI would have the information it needs
to answer correctly. If the chunk cut the answer in half, the phrase is missing and the setting scores lower.

In [7]:
# ---------- Step 4: Test questions ----------
# "q" = question, "a" = short answer phrase (lowercase) that must appear in a chunk
test_set = [
    {"q": "Who coined the term machine learning?",                      "a": "arthur samuel"},
    {"q": "Which company developed the Cybertron learning machine?",    "a": "raytheon"},
    {"q": "Which psychologist introduced the Hebbian theory?",          "a": "donald hebb"},
    {"q": "Which network improved results in the ImageNet competition?", "a": "alexnet"},
    {"q": "Who introduced generative adversarial networks?",            "a": "goodfellow"},
    {"q": "Who introduced the Transformer architecture?",               "a": "vaswani"},
    {"q": "Which fitting problem is described as a gerrymandered theory?", "a": "gerrymandered"},
    {"q": "What percentage of its original size did Chinchilla compress image data to?", "a": "43.4%"},
    {"q": "Which keyboard app uses federated learning?",                "a": "gboard"},
    {"q": "How much was the Netflix Prize grand prize?",                "a": "$1 million"},
    {"q": "Which committee demanded a full explanation for AI decisions?", "a": "house of lords"},
    {"q": "Who introduced association rules for supermarket data?",     "a": "agrawal"},
    {"q": "Which company's self-driving car failed to detect a pedestrian in 2018?", "a": "uber"},
    {"q": "How much did compute grow from AlexNet to AlphaZero?",       "a": "300,000"},
    {"q": "Which chatbot did Microsoft test that learned racist language from Twitter?", "a": "tested tay"},
    {"q": "How is data split in the holdout method?",                   "a": "2/3 training"},
    {"q": "Who proposed the first mathematical model of neural networks with McCulloch?", "a": "walter pitts"},
    {"q": "Who built the Model Inference System in 1981?",              "a": "shapiro"},
    {"q": "What is the best possible compression of x according to AIXI?", "a": "smallest possible software"},
    {"q": "Which program defeated a professional Go player in 2016?",   "a": "alphago"},
]

# ---------- Verify every answer phrase really exists in the cleaned document ----------
doc_lower = document.lower()
missing = [t for t in test_set if t["a"] not in doc_lower]

print(f"Total questions: {len(test_set)}")
if missing:
    print(f"\n⚠️ {len(missing)} answer phrase(s) NOT found in document:")
    for t in missing:
        print("   -", t["a"], "|", t["q"])
else:
    print("✅ All answer phrases found in the document")

Total questions: 20
✅ All answer phrases found in the document


# Step 6: Random Search for the Best Chunk Size and Overlap

We don't know the best settings in advance, so we let the computer try many combinations.

**How it works:**
1. We choose a **minimum and maximum** for chunk size and for overlap
2. The code picks random values inside those limits (one combination = one trial)
3. For each trial it cuts the document into chunks
4. For each of the 20 questions, it retrieves the top K chunks most similar to the question
5. It checks which of those K chunks contain the answer phrase

**Metrics (K = number of retrieved chunks):**
| Metric | Meaning | Better |
|--------|---------|--------|
| **Recall@K** | % of questions where at least one of the top K chunks contains the answer | Higher |
| **Precision@K** | Of the K chunks retrieved, what fraction contain the answer (less noise) | Higher |
| **MRR** | Rewards finding the answer in the 1st chunk more than in the 3rd | Higher |
| **Context chars** | How much text we send to the AI (cost) | Lower |

**Trade-off:** big chunks find answers easily but send a lot of text. Small chunks are cheap but can cut an answer in half.

In [ ]:
# ---------- Step 6 (SentenceTransformer): Random search ----------
# pip install sentence-transformers
import random
from sentence_transformers import SentenceTransformer

# ===== LIMITS =====
CHUNK_MIN,   CHUNK_MAX   = 200, 2000
OVERLAP_MIN, OVERLAP_MAX = 0,   400
N_TRIALS = 40
TOP_K    = 3
SEED     = 42
MODEL_NAME = "all-MiniLM-L6-v2"
# ===============================================

model = SentenceTransformer(MODEL_NAME)
model.max_seq_length = 512  # default is 256 tokens, so text after ~1000 characters gets truncated

# Embed the questions only once
q_emb = model.encode([t["q"] for t in test_set], normalize_embeddings=True)

def evaluate_st(chunk_size, overlap, top_k=TOP_K):
    chunks = fixed_size_chunking(document, chunk_size, overlap)
    c_emb = model.encode(chunks, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    sims_all = q_emb @ c_emb.T          # cosine similarity (embeddings normalized hain)
    chunks_lower = [c.lower() for c in chunks]

    recall_hits, precision_sum, rr_sum = 0, 0.0, 0.0
    for qi, t in enumerate(test_set):
        top_idx = sims_all[qi].argsort()[::-1][:top_k]
        relevant = [t["a"] in chunks_lower[i] for i in top_idx]
        if any(relevant):
            recall_hits += 1
            rr_sum += 1 / (relevant.index(True) + 1)
        precision_sum += sum(relevant) / top_k

    n = len(test_set)
    return {
        "chunks": len(chunks),
        "recall_at_k": round(recall_hits / n, 3),
        "precision_at_k": round(precision_sum / n, 3),
        "mrr": round(rr_sum / n, 3),
        "context_chars": int(top_k * np.mean([len(c) for c in chunks])),
    }

random.seed(SEED)
tried, rows = set(), []
while len(rows) < N_TRIALS:
    size = random.randrange(CHUNK_MIN, CHUNK_MAX + 1, 50)
    overlap = random.randrange(OVERLAP_MIN, min(OVERLAP_MAX, size // 2) + 1, 10)
    if (size, overlap) in tried:
        continue
    tried.add((size, overlap))
    rows.append({"chunk_size": size, "overlap": overlap, **evaluate_st(size, overlap)})
    print(f"done {len(rows)}/{N_TRIALS}: {size}/{overlap}", end="\r")

results_st = pd.DataFrame(rows).sort_values(
    ["recall_at_k", "mrr", "precision_at_k", "context_chars"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

print(f"\n🏆 SentenceTransformer ({MODEL_NAME}): Top 10 combinations (K = {TOP_K}):\n")
print(results_st.head(10).to_string(index=False))

best_st = results_st.iloc[0]
print(f"\n✅ ST Best: chunk_size={int(best_st.chunk_size)}, overlap={int(best_st.overlap)}")
print(f"   Recall@{TOP_K}={best_st.recall_at_k:.0%} | Precision@{TOP_K}={best_st.precision_at_k:.2f} | MRR={best_st.mrr}")

# ---------- Retrievers side by side comparison ----------
comp = []
for name, df in [("TF-IDF", globals().get("results")),
                 ("BM25", globals().get("results_bm25")),
                 ("SentenceTransformer", results_st)]:
    if df is not None:
        b = df.iloc[0]
        comp.append({"retriever": name, "chunk_size": int(b.chunk_size), "overlap": int(b.overlap),
                     "recall": b.recall_at_k, "precision": b.precision_at_k,
                     "mrr": b.mrr, "context_chars": b.context_chars})


Loading weights: 100%|███████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3215.87it/s]


done 38/40: 700/3400

In [ ]:
# ---------- Save BEST chunks to .txt ----------
from pathlib import Path

BEST_CHUNKS_PATH = "best_chunks.txt"

best_size, best_ovl = int(best_st.chunk_size), int(best_st.overlap)
best_chunks = fixed_size_chunking(document, best_size, best_ovl)

SEP = "=" * 80
with open(BEST_CHUNKS_PATH, "w", encoding="utf-8") as f:
    f.write(f"BEST CHUNKS | retriever: SentenceTransformer ({MODEL_NAME}) | "
            f"chunk_size={best_size}, overlap={best_ovl} | total chunks={len(best_chunks)}\n")
    f.write(SEP + "\n\n")
    for i, c in enumerate(best_chunks, 1):
        f.write(f"[CHUNK {i}/{len(best_chunks)}] ({len(c)} chars)\n")
        f.write(c + "\n\n")
        f.write("-" * 80 + "\n\n")

print(f"\n💾 {len(best_chunks)} best chunks saved -> {Path(BEST_CHUNKS_PATH).resolve()}")

# Verify: Is the number of chunks in the file correct?
saved_text = Path(BEST_CHUNKS_PATH).read_text(encoding="utf-8")
print("   chunk headers in file ->", saved_text.count("[CHUNK "), "| expected ->", len(best_chunks))

# Step 7: Validating the Best Setting on 2 Questions

Random search picked the best setting using numbers (40 random combinations, 20 test questions,
retriever: SentenceTransformer `all-MiniLM-L6-v2`). Now we **verify it visually** on 2 questions.

| Setting | Chunk size | Overlap | Meaning |
|---------|-----------|---------|---------|
| MIN | 200 | 0 | Smallest values in our search range |
| BEST | 700 | 270 | Winner of the random search (SentenceTransformer) |
| MAX | 2000 | 400 | Largest values in our search range |

For each setting we print: number of chunks, whether the answer was cut in half, the top 3 retrieved
chunks (✅ has the answer, ❌ does not), and Recall, Precision, MRR and context size.


In [ ]:
# ---------- Step 6 (SentenceTransformer): Validate TOP 3 combinations ----------
import re, textwrap
import numpy as np
import pandas as pd

# ===== SETTINGS =====
settings = {
    "TOP1": (700, 270),
    "TOP2": (500, 110),
    "TOP3": (550, 120),
    # comparison :
    "MIN":  (200, 0),
    "MAX":  (2000, 400),
}
VALIDATION_ANSWERS = ["walter pitts", "smallest possible software"]
TOP_K = 3
SHOW_FULL_CHUNK = True    # True = print full chunk 
WINDOW = 150              # Only used when SHOW_FULL_CHUNK is False
# ====================

val_set = [t for t in test_set if t["a"] in VALIDATION_ANSWERS]

# Create chunks and embeddings once for each setting
index = {}
for label, (size, ovl) in settings.items():
    chunks = fixed_size_chunking(document, size, ovl)
    c_emb = model.encode(chunks, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
    index[label] = (size, ovl, chunks, c_emb)

def highlight(text, answer):
    return re.sub(re.escape(answer), lambda m: f"【{m.group(0).upper()}】", text, flags=re.I)

def show_chunk(text, answer, full=False):
    text = highlight(text, answer)
    if not full and len(text) > 2 * WINDOW + len(answer) + 10:
        i = text.find("【")
        if i == -1:
            text = text[:2 * WINDOW] + " ..."
        else:
            text = "..." + text[max(0, i - WINDOW): i + len(answer) + 2 + WINDOW] + "..."
    return textwrap.fill(text, width=100, initial_indent="        ", subsequent_indent="        ")

scores = {label: {"recall": 0, "prec": 0.0, "rr": 0.0} for label in settings}

for n, t in enumerate(val_set, 1):
    q_vec = model.encode([t["q"]], normalize_embeddings=True)[0]

    print("#" * 100)
    print(f"QUESTION {n}: {t['q']}")
    print(f"EXPECTED ANSWER: '{t['a']}'")
    print("#" * 100)

    for label, (size, ovl, chunks, c_emb) in index.items():
        sims = c_emb @ q_vec
        top_idx = sims.argsort()[::-1][:TOP_K]
        relevant = [t["a"] in chunks[i].lower() for i in top_idx]
        survives = any(t["a"] in c.lower() for c in chunks)

        if any(relevant):
            scores[label]["recall"] += 1
            scores[label]["rr"] += 1 / (relevant.index(True) + 1)
        scores[label]["prec"] += sum(relevant) / TOP_K

        found_txt = "FOUND in top 3" if any(relevant) else "NOT found in top 3"
        print(f"\n{'=' * 100}")
        print(f">>> {label} (chunk_size={size}, overlap={ovl}) | {len(chunks)} chunks | {found_txt}"
              f" | answer cut by chunking: {'NO' if survives else 'YES'}")
        print("=" * 100)
        for rank, (i, ok) in enumerate(zip(top_idx, relevant), 1):
            print(f"\n    Rank {rank} {'✅' if ok else '❌'}  chunk #{i+1}  ({len(chunks[i])} chars)  similarity={sims[i]:.2f}")
            print(show_chunk(chunks[i], t["a"], SHOW_FULL_CHUNK))
    print()

# ---------- Comparison on the 2 validation questions ----------
n = len(val_set)
rows = []
for label, (size, ovl, chunks, c_emb) in index.items():
    rows.append({
        "setting": label, "chunk_size": size, "overlap": ovl, "chunks": len(chunks),
        f"recall@{TOP_K}": round(scores[label]["recall"] / n, 2),
        f"precision@{TOP_K}": round(scores[label]["prec"] / n, 2),
        "mrr": round(scores[label]["rr"] / n, 2),
        "context_chars": int(TOP_K * np.mean([len(c) for c in chunks])),
    })
print("📊 COMPARISON ON THE 2 VALIDATION QUESTIONS (SentenceTransformer)\n")
print(pd.DataFrame(rows).to_string(index=False))

# ---------- Same settings on ALL test questions ----------
full_rows = []
for label, (size, ovl) in settings.items():
    full_rows.append({"setting": label, "chunk_size": size, "overlap": ovl, **evaluate_st(size, ovl)})
print(f"\n📊 COMPARISON ON ALL {len(test_set)} TEST QUESTIONS (SentenceTransformer)\n")
print(pd.DataFrame(full_rows).to_string(index=False))